In [1]:
# Clone repo on a login node (as interactive nodes seem to have issues with SSH):
#   git clone git@github.com:jurgjn/euler-vibe.git
#
# Add bin/ to PATH (e.g. add to ~/.bashrc):
#   export PATH="$HOME/euler-vibe/bin:$PATH"
#
# Check that the codex-eu script is available (on an interactive node, e.g. via euler-tunnel)
which codex-eu

/cluster/home/jjaenes/euler-vibe/bin/codex-eu


In [2]:
# Build singularity image with the sandboxed codex
cd ~/euler-vibe
singularity build images/codex-eu.sif images/codex-eu.def

INFO:    User not listed in /etc/subuid, trying root-mapped namespace
INFO:    The %post section will be run under the fakeroot command
INFO:    Starting build...
INFO:    Fetching OCI image...
0.0b / 24.1MiB [------------------------------------------------] 0 % 0.0 b/s 0s
24.1MiB / 24.1MiB [===========================================] 100 % 0.0 b/s 0s
24.1MiB / 24.1MiB [===========================================] 100 % 0.0 b/s 0s
24.1MiB / 24.1MiB [===========================================] 100 % 0.0 b/s 0s
INFO:    Extracting OCI image...
INFO:    Inserting Apptainer configuration...
INFO:    Fetching OCI image...
0.0b / 28.4MiB [------------------------------------------------] 0 % 0.0 b/s 0s
0.0b / 28.4MiB [------------------------------------------------] 0 % 0.0 b/s 0s
28.4MiB / 28.4MiB [===========================================] 100 % 0.0 b/s 0s
28.4MiB / 28.4MiB [===========================================] 100 % 0.0 b/s 0s
28.4MiB / 28.4MiB [=============================

In [3]:
# codex-eu aims to act as a drop-in replacement for the vanilla codex command (except that it's sandboxed by singularity with the current directory accessible)
codex-eu --help

Executing codex with: --help
Mounting current directory: /cluster/home/jjaenes/euler-vibe
Mounting /home to: /cluster/home/jjaenes/euler-vibe/home/codex-eu
Mounting /tmp to: /scratch/tmp.63570987.jjaenes/codex-eu.1683391
Using image from: /cluster/home/jjaenes/euler-vibe/images/codex-eu.sif
Codex CLI

If no subcommand is specified, options will be forwarded to the interactive CLI.

Usage: codex [OPTIONS] [PROMPT]
       codex [OPTIONS] <COMMAND> [ARGS]

Commands:
  exec         Run Codex non-interactively [aliases: e]
  review       Run a code review non-interactively
  login        Manage login
  logout       Remove stored authentication credentials
  mcp          Manage external MCP servers for Codex
  marketplace  Manage plugin marketplaces for Codex
  mcp-server   Start Codex as an MCP server (stdio)
  app-server   [experimental] Run the app server or related tooling
  completion   Generate shell completion scripts
  sandbox      Run commands within a Codex-provided sandbox
  debug

In [4]:
# Example agentic task: check out DunbrackLab/IPSAE under $SCRATCH and optimise the ipsae.py main script
cd $SCRATCH
git clone https://github.com/DunbrackLab/IPSAE.git
cd IPSAE
codex-eu exec --no-alt-screen Please familiarise yourself with the repository and optimise ipsae.py for speed

Cloning into 'IPSAE'...
remote: Enumerating objects: 169, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 169 (delta 57), reused 51 (delta 49), pack-reused 96 (from 1)
Receiving objects: 100% (169/169), 6.43 MiB | 21.23 MiB/s, done.
Resolving deltas: 100% (74/74), done.
Executing codex with: exec --no-alt-screen Please familiarise yourself with the repository and optimise ipsae.py for speed
Mounting current directory: /cluster/scratch/jjaenes/IPSAE
Mounting /home to: /cluster/home/jjaenes/euler-vibe/home/codex-eu
Mounting /tmp to: /scratch/tmp.63570987.jjaenes/codex-eu.1683543
Using image from: /cluster/home/jjaenes/euler-vibe/images/codex-eu.sif
[>7u]10;?\]0;IPSAE\]0;⠹ IPSAE\•Booting MCP server: codex_apps(0s • esc to interrupt)›Use /skills to list available skills  gpt-5.4 default · /cluster/scratch/jjaenes/IPSAE]0;IPSAE\›Use /skills to list available skills  gpt-5.4 default · /cluster/scratch/jjaenes/IPSAEM

In [ ]:
# Show changes made by Codex
git diff

diff --git a/ipsae.py b/ipsae.py
index 77a62fc..8bbad18 100644
--- a/ipsae.py
+++ b/ipsae.py
@@ -110,7 +110,10 @@ OUT2 =             open(file2_path,'w')
 # Define the ptm and d0 functions
 def ptm_func(x,d0):
     return 1.0/(1+(x/d0)**2.0)
-ptm_func_vec=np.vectorize(ptm_func)  # vector version
+
+def ptm_func_array(x, d0):
+    x = np.asarray(x, dtype=float)
+    return 1.0 / (1.0 + (x / d0) ** 2.0)
 
 # Define the d0 functions for numbers and arrays; minimum value = 1.0; from Yang and Skolnick, PROTEINS: Structure, Function, and Bioinformatics 57:702–710 (2004)
 def calc_d0(L,pair_type):
@@ -308,23 +311,13 @@ def init_chainpairdict_set(chainlist):
     return {chain1: {chain2: set() for chain2 in chainlist if chain1 != chain2} for chain1 in chainlist}
 
 
-def classify_chains(chains, residue_types):
+def classify_chains(chains, residue_types, chain_indices):
     nuc_residue_set = {"DA", "DC", "DT", "DG", "A", "C", "U", "G"}
     chain_types = {}
 
-    # Get unique chains and itera